# Notebook 2 — K-Space Fingerprinting

---

## The Central Question

> What if you could learn something meaningful about a scan *before* reconstructing the image?

Standard MRI analysis always starts with a reconstructed image — pixels you can look at. But k-space contains that same information in raw form, encoded as frequencies. If you know how to read it, k-space can tell you things about a scan that the image alone can't.

This is the idea behind **k-space fingerprinting**: extracting a numerical signature from raw frequency data that characterizes the scan — without ever building a pixel image.

---

## Key Definitions

**Power spectrum** — the magnitude squared of k-space at each point. Tells you how much energy (signal strength) exists at each frequency. High power = that frequency is strongly represented in the tissue.

**Radial profile** — instead of looking at a full 2D grid, collapse it into a 1D curve by averaging power at each distance from the center. Distance from center = frequency. So the radial profile is: *how does signal strength change as frequency increases?*

**Energy ratio** — what fraction of the total signal lives in the low-frequency center? A high ratio means the scan is dominated by smooth, uniform tissue. A low ratio means lots of sharp edges and fine detail.

**Asymmetry score** — k-space should be roughly symmetric left-to-right for a healthy, still subject. Asymmetry can indicate motion during the scan, or structural differences between the two sides of the anatomy.

---

## Why This Matters

These three numbers — radial profile, energy ratio, asymmetry score — are features. You could:

- Feed them into a classifier to flag unusual scans before a radiologist reviews them
- Use them to detect motion artifacts automatically
- Compare fingerprints across a population to find outliers
- Track how a patient's scan changes over time without storing full images

No reconstruction required. No image processing. Just math on the raw signal.

---

## Step 1 — Import Libraries

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace, list_slices
from kode.fingerprint import radial_profile, energy_ratio, asymmetry_score

---

## Step 2 — Load the Data

We load one slice to inspect its fingerprint in detail, then loop over all slices to see how the fingerprint changes across the full 3D scan.

In [ ]:
H5_FILE = '../data/singlecoil_test/file1000022.h5'

n_slices = list_slices(H5_FILE)
print(f'This scan has {n_slices} slices (cross-sections of the knee)')

kspace = load_kspace(H5_FILE, slice_idx=15)
print(f'K-space shape: {kspace.shape}  →  (coils, height, width)')

---

## Step 3 — Plot the Radial Power Profile

This is the fingerprint curve. The x-axis is distance from the k-space center (= frequency). The y-axis is average signal power at that distance.

**What to look for:**
- The curve always drops steeply from left to right — low frequencies always dominate in MRI
- How steep the drop is, and whether there are bumps or plateaus, varies by anatomy and scan quality
- Two scans from the same anatomy type should have similar curve shapes

We use a **log scale** on the y-axis because the range of values is enormous — the center of k-space can be millions of times stronger than the edges.

In [ ]:
profile = radial_profile(kspace)

plt.figure(figsize=(10, 4))
plt.plot(profile, color='steelblue', linewidth=1.5)
plt.xlabel('Radial distance from k-space center (pixels) → increasing frequency')
plt.ylabel('Mean power (log scale)')
plt.title('K-Space Radial Power Profile — The Fingerprint Curve')
plt.yscale('log')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/fingerprint_profile.png', dpi=150)
plt.show()
print('Saved to results/fingerprint_profile.png')

---

## Step 4 — Compute Fingerprint Metrics Across All Slices

Now we loop through every slice in the scan and compute two scalar values per slice:

- **Energy ratio** — a single number between 0 and 1. Closer to 1 = more of the signal is in the low-frequency center.
- **Asymmetry score** — a single number ≥ 0. Closer to 0 = left-right symmetric k-space (expected for a clean scan).

Plotting these across slices shows how the scan's frequency content changes as you move through the knee — from one end to the other.

In [ ]:
energy_ratios = []
asymmetry_scores = []

for i in range(n_slices):
    ks = load_kspace(H5_FILE, slice_idx=i)
    energy_ratios.append(energy_ratio(ks))
    asymmetry_scores.append(asymmetry_score(ks))

print(f'Computed fingerprint for all {n_slices} slices')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(energy_ratios, color='steelblue', linewidth=1.5)
axes[0].set_title('Low-Freq Energy Ratio Across Slices\n(higher = more uniform tissue)')
axes[0].set_xlabel('Slice index (0 = one end of knee, last = other end)')
axes[0].set_ylabel('Energy ratio (0–1)')
axes[0].grid(alpha=0.3)

axes[1].plot(asymmetry_scores, color='tomato', linewidth=1.5)
axes[1].set_title('K-Space Asymmetry Score Across Slices\n(higher = more asymmetric signal)')
axes[1].set_xlabel('Slice index')
axes[1].set_ylabel('Asymmetry score')
axes[1].grid(alpha=0.3)

plt.suptitle('K-Space Fingerprint — Per-Slice Metrics', fontsize=13)
plt.tight_layout()
plt.savefig('../results/fingerprint_metrics.png', dpi=150)
plt.show()

print(f'Mean energy ratio:    {np.mean(energy_ratios):.4f}')
print(f'Mean asymmetry score: {np.mean(asymmetry_scores):.4f}')
print()
print('These two numbers are the fingerprint summary for this entire scan.')

---

## What This Means

You just extracted a compact numerical summary from a full 3D MRI scan — without ever looking at a pixel.

The real power of this approach:

- **Speed** — computing these metrics takes milliseconds; reconstructing and analyzing the image takes seconds to minutes
- **Upstream** — you're working on the raw signal before any reconstruction decisions are made
- **ML-ready** — the radial profile (a 1D vector) and the two scalar metrics are already in a format you can feed into a neural network or anomaly detector

A classifier trained on these features could flag unusual scans — motion artifacts, equipment miscalibration, anatomical outliers — before a radiologist ever opens the image.